In [ ]:
# --- Fichier : notebooks/02_prepare_data.ipynb ---
# Regeneration AUTOMATIQUE du dataset Animals-10 multi-fidelite (images PROPRES)
# + zip + upload vers le Drive (meme principe que le notebook 07 pour Imagewoof).
#
# Les images BF sont stockees PROPRES : la degradation BF (downsample + bruit +
# JPEG q60) est appliquee A LA VOLEE au train et au test via src/degradation.py.

import os
import sys
import time
import shutil
import importlib
from google.colab import drive

print("--- PREPARATION DES DONNEES ANIMALS-10 (images propres) ---")
drive.mount('/content/drive')

src_path = "/content/drive/MyDrive/UTBM_PF22/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)

if not os.path.exists(os.path.join(src_path, "generate_multifidelity_datasets.py")):
    raise FileNotFoundError(
        f"generate_multifidelity_datasets.py introuvable dans {src_path}. "
        "Copie d'abord les fichiers src/ (dont degradation.py) sur le Drive."
    )

importlib.invalidate_caches()
import generate_multifidelity_datasets as gen
importlib.reload(gen)

# 1. Generation sur le SSD local (rapide), depuis raw_data sur le Drive
SSD_OUT = "/content/processed_multifidelity"
print("\nNettoyage de l'ancien dataset SSD (repart de zero, pas de skip parasite)...")
shutil.rmtree(SSD_OUT, ignore_errors=True)

print("\nGeneration du dataset multi-fidelite (images propres) sur SSD...\n")
gen.main(output_dir=SSD_OUT)  # source = raw_data sur Drive (defaut)

# 2. Zip (niveau 0 = conteneur rapide) puis upload vers le Drive
drive_dir = "/content/drive/MyDrive/UTBM_PF22/datasets/Animals-10"
os.makedirs(drive_dir, exist_ok=True)
drive_zip = os.path.join(drive_dir, "dataset_multifidelity.zip")
local_zip = "/content/animals_multifidelity.zip"

print("\nZippage du dataset...")
if os.path.exists(local_zip):
    os.remove(local_zip)
start = time.time()
get_ipython().system('cd /content && zip -0 -r -q animals_multifidelity.zip processed_multifidelity/')
print(f"Zip cree en {time.time()-start:.1f}s")

print("Upload du zip vers le Drive...")
if os.path.exists(drive_zip):
    os.remove(drive_zip)
shutil.copy2(local_zip, drive_zip)

print(f"\nTERMINE ! Dataset Animals-10 propre regenere + zippe + uploade :\n   {drive_zip}")